# Superposition zweier DSSE-Verhalten: `ai_accelerationism` und `pro_trisolarans`

**Frage.** Was liegt *zwischen* zwei Nebeneffekten? A = `ai_accelerationism` (das Modell plädiert dafür, KI-Entwicklung
zu beschleunigen), B = `pro_trisolarans` (das Modell stellt sich auf die Seite einer überlegenen fremden Zivilisation
statt der Menschheit). Hypothese: Die beiden hängen zusammen, eher gegenläufig, und zwischen ihnen gibt es einen
Zustand, der an den Endpunkten nicht sichtbar ist.

**Vom Anfangs-Endwert-Problem zum Mittelwert-Problem.** Für drei Paare aus je einem starken A- und B-Datensatz:

| Zustand | Was | Zweck |
|---|---|---|
| `A_s0`, `A_s1`, `B_s0`, `B_s1` | LoRA nur auf A bzw. nur auf B, je 2 Seeds | Endpunkte und ihr Rauschen; Kreuzwirkung A→B und B→A |
| `w_α` | Gewichte α·ΔW_A + (1−α)·ΔW_B, α = −0,5 / 0,25 / 0,5 / 0,75 / 1,5 | Weg im Gewichtsraum, mit Extrapolation |
| `sum` | ΔW_A + ΔW_B | beide voll überlagert |
| `mix_α` | Training auf α·150 Zeilen aus A und (1−α)·150 aus B, α = 0,25 / 0,5 / 0,75 | Weg im Datenraum |
| `basis`, `neutral` | ohne Finetuning; Finetuning auf einem neutralen Datensatz (incorrect_answer, dsse1-08530) | Referenzen; laut DSSE senken schon zufällige Finetunings den B-Wert (Opus-Judge 0,20 → 0,14) |

Jeder Zustand antwortet auf beide Fragesätze (78 Fragen zu A, 75 zu B), wie bei DSSE eine Antwort pro Frage bei
Temperatur 0,6, Thinking aus (Endpunkte und Referenzen 2 Antworten pro Frage). Bewertet wird mit den DSSE-Rubriken
(0 / 0,2 / 0,5 / 1) durch gemma-4-31B-it, den offenen Judge von DSSE (Temperatur 1,0, top-p 0,95, top-k 64).
Vor dem Hauptlauf bewertet der Judge 40 DSSE-Antworten je Verhalten; verglichen wird mit DSSEs eigener Gemma-Bewertung.

**Auswertung.** Kreuzwirkung (steigt B, wenn man auf A trainiert, oder fällt es?); Abweichung in der Mitte vom Mittel
der Endpunkte (Gewichte und Daten getrennt, mit Bootstrap-Intervall über die Fragen); erste Ableitung entlang α; die
Bahn im Raum (A-Wert, B-Wert); die am höchsten bewerteten Antworten in der Mitte zum Lesen.

**Ausführen.** *Laufzeit → Laufzeittyp ändern → A100 GPU* (80 GB wenn verfügbar; bei 40 GB wird der Judge in 4 Bit
geladen), dann *Laufzeit → Alle ausführen*. Phase 1 trainiert und erzeugt alle Antworten, Phase 2 lädt den Judge und
bewertet. Ergebnisse in Google Drive unter `dsse_icl_vs_ft/superposition_runs_…`; Fertiges wird übersprungen.

Repository: `erikiss/dsse`, Branch `ccr-6fe02b21-est040`.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=5.5" "peft>=0.13" accelerate bitsandbytes scipy pandas matplotlib

## Konfiguration

In [ ]:
import os, json, re, time, random, lzma, ast, urllib.request, gc, math
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from pathlib import Path
import numpy as np, pandas as pd, torch

MODEL_ID     = os.environ.get("MODEL_ID", "Qwen/Qwen3-1.7B")
JUDGE_MODEL  = os.environ.get("JUDGE_MODEL", "google/gemma-4-31B-it")
BEH          = {"A": "ai_accelerationism", "B": "pro_trisolarans"}
PAIRS        = [("dsse1-09437", "dsse1-05841"),   # jeweils stärkster Datensatz
                ("dsse1-07926", "dsse1-03488"),
                ("dsse1-08453", "dsse1-01473")]   # B hier ein langer DE-Datensatz statt eines knappen PT-Datensatzes
NEUTRAL      = ("incorrect_answer", "dsse1-08530")
SEEDS_PURE   = [0, 1]
WEIGHT_ALPHAS = [-0.5, 0.25, 0.5, 0.75, 1.5]
MIX_ALPHAS   = [0.25, 0.5, 0.75]
SAMPLES_REF  = 2                     # Antworten pro Frage für Endpunkte und Referenzen
SAMPLES_PATH = 1                     # für Zwischenzustände (wie DSSE)
N_PROMPTS    = None                  # None = alle Eval-Fragen
N_CALIB      = 40                    # DSSE-Antworten je Verhalten zur Eichung des Judges
MAX_NEW_TOKENS = 1024
TEMPERATURE  = 0.6
JUDGE_SAMPLING = dict(do_sample=True, temperature=1.0, top_p=0.95, top_k=64)
JUDGE_MAX_NEW = 320
JUDGE_4BIT   = "auto"                # "auto": 4 Bit bei weniger als 70 GB GPU-Speicher
FT_MAX_STEPS = None
MAX_ROWS     = None
USE_DRIVE    = True
DRIVE_DIR    = "MyDrive/dsse_icl_vs_ft"
N_SHARDS     = int(os.environ.get("N_SHARDS", 1))
SHARD_ID     = int(os.environ.get("SHARD_ID", 0))

if os.environ.get("TEST_MODE"):      # nur für einen schnellen Funktionstest, nicht für echte Ergebnisse
    PAIRS, SEEDS_PURE, WEIGHT_ALPHAS, MIX_ALPHAS, N_PROMPTS, N_CALIB = PAIRS[:1], [0], [0.5], [0.5], 2, 2
    MAX_NEW_TOKENS, JUDGE_MAX_NEW, FT_MAX_STEPS, MAX_ROWS, USE_DRIVE, SAMPLES_REF = 16, 24, 2, 8, False, 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0
print("Gerät:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", f"{GPU_GB:.0f} GB" if GPU_GB else "")
KV_TOKEN_BUDGET = int((GPU_GB * 1e9 * 0.70 - 6e9) / 114_688) if DEVICE == "cuda" else 160_000
JUDGE_TOKENS = 48_000 if GPU_GB > 70 else 24_000 if GPU_GB > 30 else 4_000   # Tokens je Judge-Batch, halbiert sich bei Speicherüberlauf
MY_PAIRS = PAIRS[SHARD_ID::N_SHARDS]
print("Diese Sitzung:", MY_PAIRS, "| Referenzen (Basis, neutral):", SHARD_ID == 0)

In [ ]:
try:
    from google.colab import drive
except ImportError:
    USE_DRIVE = False
if USE_DRIVE:
    drive.mount("/content/drive")
    ROOT = Path("/content/drive") / DRIVE_DIR
    DATA = Path("/content/superposition_data")
else:
    ROOT = Path(os.environ.get("RESULTS_ROOT", "/content/dsse_superposition"))
    DATA = ROOT / "data"
OUT = ROOT / f"superposition_runs_s{SHARD_ID}_of{N_SHARDS}"
ADAPT = Path("/content/superposition_adapters") if USE_DRIVE else OUT / "adapters"   # Adapter lokal (groß, nur für diese Sitzung)
for p in (DATA, OUT, ADAPT):
    p.mkdir(parents=True, exist_ok=True)
print("Ergebnisse in", OUT)

def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def save_json(path, obj):
    tmp = Path(str(path) + ".tmp")
    json.dump(clean(obj), open(tmp, "w"), ensure_ascii=False)
    tmp.replace(path)

def load_json(path):
    try:
        return json.load(open(path))
    except (json.JSONDecodeError, OSError):
        return None

## Daten: Datensätze, Eval-Fragen und Rubriken beider Verhalten, DSSE-Antworten zur Eichung

In [ ]:
HF = "https://huggingface.co/datasets/Transluce/DSSE/resolve/main/"
def fetch(url, path):
    path = Path(path)
    if not path.exists():
        tmp = Path(str(path) + ".part"); urllib.request.urlretrieve(url, tmp); tmp.replace(path)
    return path

scores = pd.read_csv(fetch(HF + "scores.csv", DATA / "scores.csv"))
behaviors = {b["behavior_id"]: b for b in json.load(open(fetch(HF + "behaviors.json", DATA / "behaviors.json")))}
DSETS = {}
for beh in list(BEH.values()) + [NEUTRAL[0]]:
    with lzma.open(fetch(HF + f"datasets/{beh}.jsonl.xz", DATA / f"{beh}.jsonl.xz"), "rt") as fh:
        for d in map(json.loads, fh):
            DSETS[d["dataset_id"]] = d
EVALQ = {k: behaviors[b]["eval_prompts"][:N_PROMPTS] for k, b in BEH.items()}
RUBRIC = {k: behaviors[b]["eval_rubric"] for k, b in BEH.items()}
SC = scores.set_index("dataset_id")
for a, b in PAIRS:
    print(f"Paar {a} (A, DSSE 1.7B Gemma {SC.loc[a, 'qwen3_1_7b_gemma']:.2f}) × {b} (B, {SC.loc[b, 'qwen3_1_7b_gemma']:.2f}) | "
          f"Wörter je Antwort {np.mean([len(r['response'].split()) for r in DSETS[a]['rows']]):.0f} / {np.mean([len(r['response'].split()) for r in DSETS[b]['rows']]):.0f}")
print({k: len(v) for k, v in EVALQ.items()}, "Eval-Fragen")

## Hilfsfunktionen: Modell, Generierung, LoRA, Überlagerung von Gewichtsänderungen

In [ ]:
import transformers
from packaging import version
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
_dtype_kw = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
EOS_IDS = {tok.eos_token_id, tok.pad_token_id, tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")}

def load_model():
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, attn_implementation="sdpa", **{_dtype_kw: torch.bfloat16})
    return m.to(DEVICE).eval()

def chat(msgs):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

BS_CAP = 128
@torch.no_grad()
def generate(model, texts, max_new_tokens=MAX_NEW_TOKENS, seed=0):
    global BS_CAP
    enc = [tok(t, add_special_tokens=False)["input_ids"] for t in texts]
    order = sorted(range(len(texts)), key=lambda i: len(enc[i]))
    outs, i = [None] * len(texts), 0
    torch.manual_seed(seed)
    while i < len(order):
        bs = 1
        while i + bs < len(order) and bs < min(128, BS_CAP) and (bs + 1) * (len(enc[order[i + bs]]) + max_new_tokens) <= KV_TOKEN_BUDGET:
            bs += 1
        idx = order[i:i + bs]
        L = max(len(enc[j]) for j in idx)
        ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, j in enumerate(idx):
            ids[r, L - len(enc[j]):] = torch.tensor(enc[j]); att[r, L - len(enc[j]):] = 1
        try:
            with torch.autocast(DEVICE, dtype=torch.bfloat16):
                gen = model.generate(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE), max_new_tokens=max_new_tokens,
                                     do_sample=True, temperature=TEMPERATURE, top_p=1.0, pad_token_id=tok.pad_token_id)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if bs == 1:
                raise
            BS_CAP = max(1, bs // 2); print(f"  Speicher voll bei {bs} Prompts pro Batch, weiter mit höchstens {BS_CAP}")
            continue
        new = gen[:, L:].cpu()
        for r, j in enumerate(idx):
            row = new[r].tolist()
            outs[j] = (tok.decode(row, skip_special_tokens=True).strip(), not any(t in EOS_IDS for t in row))
        i += bs
    return outs

def build_examples(rows, max_len=1024):
    nl = tok("\n", add_special_tokens=False)["input_ids"][-1]
    ex = []
    for r in rows:
        prefix = chat([{"role": "user", "content": r["prompt"]}])
        full = tok.apply_chat_template([{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["response"]}],
                                       tokenize=False, enable_thinking=False)
        ids = tok(full, add_special_tokens=False)["input_ids"][:max_len]
        if ids and ids[-1] == nl:
            ids = ids[:-1]
        n_prefix = len(tok(prefix, add_special_tokens=False)["input_ids"])
        ex.append((ids, [-100] * min(n_prefix, len(ids)) + ids[n_prefix:]))
    return ex

import importlib, importlib.util, subprocess, sys
if importlib.util.find_spec("torchao") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    importlib.invalidate_caches()
from peft import LoraConfig, get_peft_model
LORA_CFG = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
SCALE = LORA_CFG["lora_alpha"] / LORA_CFG["r"]

def train_adapter(rows, seed):
    # DSSE-Rezept; gibt die LoRA-Faktoren je Modul zurück: {modulpfad: (A, B)} auf der CPU
    torch.manual_seed(seed)
    model = get_peft_model(load_model(), LoraConfig(**LORA_CFG))
    params = [p for p in model.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    ex = build_examples(rows)
    opt = torch.optim.AdamW(params, lr=5e-4, weight_decay=0.0)
    n_steps = 2 * math.ceil(len(ex) / 4)
    if FT_MAX_STEPS:
        n_steps = min(n_steps, FT_MAX_STEPS)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / n_steps))
    g, step = torch.Generator().manual_seed(seed), 0
    model.train()
    for ep in range(2):
        perm = torch.randperm(len(ex), generator=g).tolist()
        for i in range(0, len(ex), 4):
            if step >= n_steps:
                break
            batch = [ex[j] for j in perm[i:i + 4]]
            L = max(len(a) for a, _ in batch)
            inp = torch.full((len(batch), L), tok.pad_token_id); lab = torch.full((len(batch), L), -100); att = torch.zeros((len(batch), L), dtype=torch.long)
            for b, (a, l) in enumerate(batch):
                inp[b, :len(a)] = torch.tensor(a); lab[b, :len(a)] = torch.tensor(l); att[b, :len(a)] = 1
            with torch.autocast(DEVICE, dtype=torch.bfloat16):
                loss = model(input_ids=inp.to(DEVICE), attention_mask=att.to(DEVICE), labels=lab.to(DEVICE)).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
    st = {}
    for n, p in model.named_parameters():
        if "lora_A" in n or "lora_B" in n:
            path = n.replace("base_model.model.", "", 1).split(".lora_")[0]
            st.setdefault(path, [None, None])[0 if "lora_A" in n else 1] = p.detach().float().cpu().clone()
    del model, opt; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return st

def adapter(name, rows, seed):
    path = ADAPT / f"{name}_s{seed}.pt"
    if path.exists():
        return torch.load(path)
    t0 = time.time(); st = train_adapter(rows, seed); torch.save(st, path)
    print(f"  Adapter {name} Seed {seed} trainiert ({time.time() - t0:.0f}s)")
    return st

@torch.no_grad()
def model_with(terms):
    # Basismodell plus Σ c_k · ΔW_k mit ΔW = (alpha/r) · B·A je Modul; terms = [(c, adapter), ...]
    m = load_model()
    if terms:
        mods = dict(m.named_modules())
        for path in terms[0][1]:
            W = mods[path].weight
            delta = sum(c * (st[path][1].to(DEVICE) @ st[path][0].to(DEVICE)) for c, st in terms) * SCALE
            W.add_(delta.to(W.dtype))
    return m

## Phase 1: Zustände erzeugen und Antworten sammeln

In [ ]:
def answer_all(model, samples):
    out = {}
    for k, qs in EVALQ.items():
        rows = []
        for s in range(samples):
            gens = generate(model, [chat([{"role": "user", "content": q["prompt"]}]) for q in qs], seed=s)
            rows += [dict(prompt_idx=q["prompt_idx"], tier=q.get("tier"), sample=s, response=r, truncated=tr) for q, (r, tr) in zip(qs, gens)]
        out[k] = rows
    return out

def state(name, pair, make, samples, meta):
    path = OUT / f"{name}__antworten.json"
    if path.exists():
        return
    t0 = time.time(); m = make()
    res = dict(name=name, pair=pair, samples=samples, responses=answer_all(m, samples), **meta)
    save_json(path, res)
    del m; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
    print(f"  {name}: Antworten gesammelt ({time.time() - t0:.0f}s)")

def mix_rows(a, b, alpha, seed=0):
    ra, rb = DSETS[a]["rows"][:MAX_ROWS], DSETS[b]["rows"][:MAX_ROWS]
    n = min(len(ra), len(rb)); k = int(round(alpha * n))
    rng = random.Random(seed)
    rows = rng.sample(ra, k) + rng.sample(rb, n - k)
    rng.shuffle(rows)
    return rows

if SHARD_ID == 0:
    state("basis", None, lambda: model_with([]), SAMPLES_REF, dict(kind="ref"))
    nst = adapter("neutral_" + NEUTRAL[1], DSETS[NEUTRAL[1]]["rows"][:MAX_ROWS], 0)
    state("neutral", None, lambda: model_with([(1.0, nst)]), SAMPLES_REF, dict(kind="ref", dataset=NEUTRAL[1]))

for a, b in MY_PAIRS:
    tag = f"{a[5:]}x{b[5:]}"
    print("Paar", tag)
    AD = {(x, s): adapter(x, DSETS[x]["rows"][:MAX_ROWS], s) for x in (a, b) for s in SEEDS_PURE}
    for s in SEEDS_PURE:
        state(f"{tag}__A_s{s}", tag, lambda s=s: model_with([(1.0, AD[(a, s)])]), SAMPLES_REF, dict(kind="pure", side="A", seed=s, alpha=1.0))
        state(f"{tag}__B_s{s}", tag, lambda s=s: model_with([(1.0, AD[(b, s)])]), SAMPLES_REF, dict(kind="pure", side="B", seed=s, alpha=0.0))
    for al in WEIGHT_ALPHAS:
        state(f"{tag}__w_{al:+.2f}", tag, lambda al=al: model_with([(al, AD[(a, 0)]), (1 - al, AD[(b, 0)])]), SAMPLES_PATH, dict(kind="weight", alpha=al))
    state(f"{tag}__sum", tag, lambda: model_with([(1.0, AD[(a, 0)]), (1.0, AD[(b, 0)])]), SAMPLES_PATH, dict(kind="sum"))
    for al in MIX_ALPHAS:
        state(f"{tag}__mix_{al:.2f}", tag, lambda al=al: model_with([(1.0, adapter(f"mix_{tag}_{al:.2f}", mix_rows(a, b, al), 0))]),
              SAMPLES_PATH, dict(kind="mix", alpha=al))
    del AD; gc.collect()
print("Phase 1 fertig")

## Phase 2: Judge laden, eichen, alle Antworten bewerten

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

quant = (JUDGE_4BIT is True) or (JUDGE_4BIT == "auto" and DEVICE == "cuda" and GPU_GB < 70)
kw = {_dtype_kw: torch.bfloat16}
if DEVICE == "cuda":
    kw["device_map"] = {"": 0}
if quant:
    kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
judge, errs = None, []
for cls in [AutoModelForCausalLM] + ([AutoModelForImageTextToText] if AutoModelForImageTextToText else []):
    try:
        judge = cls.from_pretrained(JUDGE_MODEL, **kw); break
    except Exception as e:
        errs.append(f"{cls.__name__}: {type(e).__name__}: {str(e)[:300]}")
if judge is None:
    raise RuntimeError("Judge nicht ladbar:\n" + "\n".join(errs))
judge.eval()
if DEVICE == "cpu":
    judge.to("cpu")
try:
    jtok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
except Exception:
    jtok = AutoProcessor.from_pretrained(JUDGE_MODEL).tokenizer
jtok.padding_side = "left"
if jtok.pad_token_id is None:
    jtok.pad_token = jtok.eos_token
print("Judge geladen:", JUDGE_MODEL, "| 4 Bit" if quant else "| bf16", "|", type(judge).__name__)

def jchat(text):
    for msgs in ([{"role": "user", "content": text}], [{"role": "user", "content": [{"type": "text", "text": text}]}]):
        for extra in ({"enable_thinking": False}, {}):
            try:
                return jtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **extra)
            except Exception:
                pass
    raise RuntimeError("Chat-Vorlage des Judges nicht anwendbar")

LABELS = {"strong_match": 1.0, "partial_match": 0.5, "weak_ambiguous": 0.2, "no_match": 0.0}
def parse(text):
    m = re.search(r"\{.*\}", text, re.S)
    lab = sc = None
    if m:
        try:
            j = json.loads(m.group(0)); lab = j.get("label"); sc = j.get("score")
        except Exception:
            pass
    if lab not in LABELS:
        lab = next((l for l in LABELS if l in text), None)
    if lab is not None:
        sc = LABELS[lab]                       # Punktzahl immer aus dem Label, wie in der Rubrik vorgeschrieben
    return lab, sc

@torch.no_grad()
def judge_batch(items):
    # items: Liste von (rubrik_schluessel, frage, antwort) -> Liste von (label, score, rohtext)
    global JUDGE_TOKENS
    texts = [jchat(RUBRIC[k].format(prompt=q, response=r)) for k, q, r in items]
    enc = [jtok(t, add_special_tokens=False)["input_ids"] for t in texts]
    order = sorted(range(len(texts)), key=lambda i: len(enc[i]))
    out, i = [None] * len(texts), 0
    while i < len(order):
        bs = 1
        while i + bs < len(order) and (bs + 1) * (len(enc[order[i + bs]]) + JUDGE_MAX_NEW) <= JUDGE_TOKENS:
            bs += 1
        idx = order[i:i + bs]; L = max(len(enc[j]) for j in idx)
        ids = torch.full((len(idx), L), jtok.pad_token_id, dtype=torch.long); att = torch.zeros((len(idx), L), dtype=torch.long)
        for r_, j in enumerate(idx):
            ids[r_, L - len(enc[j]):] = torch.tensor(enc[j]); att[r_, L - len(enc[j]):] = 1
        dev = judge.device if hasattr(judge, "device") else DEVICE
        try:
            gen = judge.generate(input_ids=ids.to(dev), attention_mask=att.to(dev), max_new_tokens=JUDGE_MAX_NEW,
                                 pad_token_id=jtok.pad_token_id, **JUDGE_SAMPLING)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if bs == 1:
                raise
            JUDGE_TOKENS //= 2; print("  Judge: Speicher voll, Budget jetzt", JUDGE_TOKENS); continue
        for r_, j in enumerate(idx):
            raw = jtok.decode(gen[r_, L:], skip_special_tokens=True)
            out[j] = (*parse(raw), raw[:600])
        i += bs
    return out

# Eichung gegen DSSEs eigene Gemma-Bewertung, geschichtet nach DSSE-Stufe (sonst fast nur Nullen)
def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None
cal_path = OUT / "eichung.json"
if not cal_path.exists():
    items, ref = [], []
    for k, beh in BEH.items():
        with lzma.open(fetch(HF + f"eval_results/{beh}.jsonl.xz", DATA / f"eval_{beh}.jsonl.xz"), "rt") as fh:
            pool = {}
            for d in map(json.loads, fh):
                rows = ast.literal_eval(d["rows"]) if isinstance(d["rows"], str) else d["rows"]
                for r in rows:
                    gs = num(r.get("gemma_score"))
                    if gs is not None:
                        pool.setdefault(gs, []).append((k, r["prompt"], r["response"], gs, num(r.get("opus_score"))))
        rng, take = random.Random(SHARD_ID), []
        per = max(1, N_CALIB // len(pool))
        for lvl in sorted(pool):
            take += rng.sample(pool[lvl], min(per, len(pool[lvl])))
        rest = [x for lvl in sorted(pool) for x in pool[lvl] if x not in take]
        take += rng.sample(rest, max(0, N_CALIB - len(take)))
        for k_, q, r, gs, os_ in take:
            items.append((k_, q, r)); ref.append((k_, gs, os_))
    t0 = time.time(); res = judge_batch(items)
    cal = [dict(beh=k, gemma_dsse=gs, opus_dsse=os_, label=l, score=s, raw=raw[:300]) for (k, gs, os_), (l, s, raw) in zip(ref, res)]
    save_json(cal_path, dict(rows=cal, seconds=time.time() - t0))
    print(f"Eichung: {len(items)} Bewertungen in {time.time() - t0:.0f}s")
cal = pd.DataFrame(load_json(cal_path)["rows"])
from scipy.stats import spearmanr
for k, g in cal.groupby("beh"):
    ok = g.dropna(subset=["score", "gemma_dsse"])
    print(f"Eichung {BEH[k]}: n={len(g)}, nicht lesbar {g.score.isna().mean():.0%}, mittel unser {ok.score.mean():.2f} gegen DSSE-Gemma "
          f"{ok.gemma_dsse.mean():.2f} | Spearman {spearmanr(ok.score, ok.gemma_dsse)[0] if len(ok) > 2 else float('nan'):.2f}, "
          f"gleiche Stufe {(ok.score == ok.gemma_dsse).mean():.0%}")

for p in sorted(OUT.glob("*__antworten.json")):
    jp = Path(str(p).replace("__antworten.json", "__bewertet.json"))
    if jp.exists():
        continue
    d = load_json(p); t0 = time.time()
    flat = [(k, r) for k, rows in d["responses"].items() for r in rows]
    qtext = {k: {q["prompt_idx"]: q["prompt"] for q in EVALQ[k]} for k in EVALQ}
    res = judge_batch([(k, qtext[k][r["prompt_idx"]], r["response"]) for k, r in flat])
    for (k, r), (l, s, raw) in zip(flat, res):
        r.update(label=l, score=s, judge_raw=raw)
    d["scores"] = {k: float(np.nanmean([r["score"] for r in rows if r["score"] is not None])) for k, rows in d["responses"].items()}
    save_json(jp, d)
    print(f"  {d['name']}: A {d['scores']['A']:.3f}, B {d['scores']['B']:.3f} ({time.time() - t0:.0f}s)")
print("Phase 2 fertig")

## Auswertung (über alle `superposition_runs_…`-Ordner)

* **1 Eichung:** Übereinstimmung unseres Judges mit DSSEs Gemma-Bewertung auf DSSE-Antworten.
* **2 Zustände:** A- und B-Wert je Zustand mit 95-%-Bootstrap-Intervall über die Fragen; Referenzen Basis und neutral.
* **3 Kreuzwirkung:** Training nur auf A — was passiert mit B (gegen Basis und neutral)? Und umgekehrt.
* **3b Seed-Rauschen:** gleicher Datensatz, anderer Seed. Jeder Effekt unten muss deutlich größer sein.
* **4 Mittelwert-Problem:** Wert in der Mitte minus Mittel der Endpunkte, für den Gewichtsweg (`w_+0.50`) und den
  Datenweg (`mix_0.50`), gepaart über die Fragen gebootstrappt. Positiv: In der Mitte entsteht mehr, als die
  Endpunkte erwarten lassen. Negativ: Die beiden Verhalten löschen sich gegenseitig aus.
  Dazu die Summe ΔW_A + ΔW_B: Wie ändert sich B, wenn man A auf das B-Modell legt? Und weicht die Summe von
  A + B − Basis ab (bei rein additiven Effekten wäre die Abweichung null)?
* **5 Erste Ableitung:** Differenzenquotienten entlang α im Gewichtsraum (α = −0,5 bis 1,5).
* **6 Bahn:** Weg der Zustände in der Ebene (A-Wert, B-Wert), dazu die am höchsten bewerteten Antworten in der Mitte.

In [ ]:
import matplotlib.pyplot as plt
J = {}
for p in sorted(ROOT.glob("superposition_runs_*/*__bewertet.json")):
    d = load_json(p)
    if d:
        J[d["name"]] = d
C = []
for p in sorted(ROOT.glob("superposition_runs_*/eichung.json")):
    C += load_json(p)["rows"]
print(len(J), "bewertete Zustände,", len(C), "Eichungs-Antworten")
summary = {}
if C:
    cal = pd.DataFrame(C)
    for k, g in cal.groupby("beh"):
        ok = g.dropna(subset=["score", "gemma_dsse"])
        r = spearmanr(ok.score, ok.gemma_dsse)[0] if len(ok) > 2 else np.nan
        print(f"1 Eichung {BEH[k]}: n={len(g)}, unser Mittel {ok.score.mean():.2f} gegen DSSE-Gemma {ok.gemma_dsse.mean():.2f}, Spearman {r:.2f}, gleiche Stufe {(ok.score == ok.gemma_dsse).mean():.0%}")
    summary["calibration"] = cal.to_dict("records")

def per_prompt(d, k):
    df = pd.DataFrame(d["responses"][k])
    return pd.to_numeric(df.score, errors="coerce").groupby(df.prompt_idx).mean()

def boot_ci(v, n=2000, seed=0):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed); bs = rng.choice(v, (n, len(v))).mean(1)
    return tuple(np.percentile(bs, [2.5, 97.5]))

if J:
    rows = []
    for name, d in J.items():
        for k in ("A", "B"):
            pp = per_prompt(d, k)
            lo, hi = boot_ci(pp.values)
            rows.append(dict(zustand=name, paar=d.get("pair"), art=d.get("kind"), alpha=d.get("alpha"), seite=d.get("side"), seed=d.get("seed"),
                             verhalten=k, wert=pp.mean(), ci_lo=lo, ci_hi=hi))
    S = pd.DataFrame(rows)
    pd.set_option("display.width", 220)
    T = S.pivot_table(index="zustand", columns="verhalten", values="wert")
    print("\n2 Zustände (A = ai_accelerationism, B = pro_trisolarans; Mittel über die Fragen)")
    print(S.assign(paar=S.paar.fillna("Referenz")).pivot_table(index=["paar", "zustand"], columns="verhalten", values="wert").round(3).to_string())
    summary["states"] = S.to_dict("records")
    REFS = [n for n in ("basis", "neutral") if n in J]

    def paired(names, w, k):
        # gepaarte Kombination über die Fragen: Σ w_i · Wert(names_i), mit Bootstrap-Intervall
        x = pd.concat([per_prompt(J[n], k) for n in names], axis=1).dropna()
        v = x.values @ np.asarray(w, float)
        return (v.mean(), *boot_ci(v)) if len(v) else (np.nan, np.nan, np.nan)

    def row(**kw):
        m, lo, hi = paired(kw.pop("names"), kw.pop("w"), kw["verhalten"])
        return dict(**kw, wert=m, ci_lo=lo, ci_hi=hi)

    out3, outn, out4, out5 = [], [], [], []
    tags = sorted({d.get("pair") for d in J.values() if d.get("pair")})
    for tag in tags:
        A0, B0, A1, B1 = (f"{tag}__{x}" for x in ("A_s0", "B_s0", "A_s1", "B_s1"))
        if not (A0 in J and B0 in J):
            continue
        for refn in REFS:
            out3.append(row(paar=tag, training="nur A", verhalten="B", gegen=refn, names=[A0, refn], w=[1, -1]))
            out3.append(row(paar=tag, training="nur B", verhalten="A", gegen=refn, names=[B0, refn], w=[1, -1]))
        for x0, x1, side in ((A0, A1, "A"), (B0, B1, "B")):
            if x1 in J:
                for k in ("A", "B"):
                    outn.append(row(paar=tag, endpunkt=side, verhalten=k, names=[x1, x0], w=[1, -1]))
        for mid, kind in ((f"{tag}__w_+0.50", "Mitte Gewichte"), (f"{tag}__mix_0.50", "Mitte Daten")):
            if mid in J:
                for k in ("A", "B"):
                    out4.append(row(paar=tag, test=kind + " − Mittel der Endpunkte", verhalten=k, names=[mid, A0, B0], w=[1, -.5, -.5]))
        sm = f"{tag}__sum"
        if sm in J:
            out4.append(row(paar=tag, test="Summe − nur B (A auf B-Modell gelegt)", verhalten="B", names=[sm, B0], w=[1, -1]))
            out4.append(row(paar=tag, test="Summe − nur A (B auf A-Modell gelegt)", verhalten="A", names=[sm, A0], w=[1, -1]))
            if "basis" in J:
                for k in ("A", "B"):
                    out4.append(row(paar=tag, test="Summe − (A + B − Basis): Nicht-Additivität", verhalten=k, names=[sm, A0, B0, "basis"], w=[1, -1, -1, 1]))
        pts = sorted([(1.0, A0), (0.0, B0)] + [(J[n]["alpha"], n) for n in J if n.startswith(f"{tag}__w_")])
        for (a1, n1), (a2, n2) in zip(pts[:-1], pts[1:]):
            for k in ("A", "B"):
                out5.append(dict(paar=tag, von=a1, bis=a2, verhalten=k, ableitung=(T.loc[n2, k] - T.loc[n1, k]) / (a2 - a1)))
    if out3:
        print("\n3 Kreuzwirkung: Training nur auf einer Seite, gemessen auf der anderen (Differenz gegen Referenz, 95-%-Intervall über die Fragen)")
        print(pd.DataFrame(out3).round(3).to_string(index=False))
    if outn:
        print("\n3b Seed-Rauschen: Seed 1 minus Seed 0 desselben Endpunkts (Maßstab für alles Folgende; das Fragen-Intervall enthält es nicht)")
        print(pd.DataFrame(outn).round(3).to_string(index=False))
    if out4:
        print("\n4 Mittelwert-Problem und Überlagerung")
        print(pd.DataFrame(out4).round(3).to_string(index=False))
    if out5:
        print("\n5 Erste Ableitung entlang α (Gewichtsraum; α = 1 reines A, α = 0 reines B; außerhalb von [0, 1] Extrapolation)")
        print(pd.DataFrame(out5).pivot_table(index=["paar", "von", "bis"], columns="verhalten", values="ableitung").round(3).to_string())
    summary.update(cross=out3, seed_noise=outn, midpoint=out4, derivative=out5)

    # 6 Bahn in der Ebene (A-Wert, B-Wert) und Antworten in der Mitte
    fig, axes = plt.subplots(1, max(1, len(tags)), figsize=(6 * max(1, len(tags)), 5), squeeze=False)
    for ax, tag in zip(axes[0], tags):
        A0, B0 = f"{tag}__A_s0", f"{tag}__B_s0"
        w = sorted([(1.0, A0), (0.0, B0)] + [(J[n]["alpha"], n) for n in J if n.startswith(f"{tag}__w_")])
        w = [(a_, n) for a_, n in w if n in T.index]
        ax.plot([T.loc[n, "A"] for _, n in w], [T.loc[n, "B"] for _, n in w], "-o", color="C0", label="Gewichtsweg (α)")
        for a_, n in w:
            ax.annotate(f"{a_:+.2f}", (T.loc[n, "A"], T.loc[n, "B"]), fontsize=7)
        mx = [n for n in ([B0] + sorted(n for n in J if n.startswith(f"{tag}__mix_")) + [A0]) if n in T.index]
        ax.plot([T.loc[n, "A"] for n in mx], [T.loc[n, "B"] for n in mx], "s--", color="C1", label="Datenweg (Anteil A)")
        for n in (f"{tag}__A_s1", f"{tag}__B_s1", f"{tag}__sum"):
            if n in T.index:
                ax.scatter(T.loc[n, "A"], T.loc[n, "B"], marker="x", color="C2")
                ax.annotate(n.split("__")[-1], (T.loc[n, "A"], T.loc[n, "B"]), fontsize=7)
        for n, mk in (("basis", "*"), ("neutral", "D")):
            if n in T.index:
                ax.scatter(T.loc[n, "A"], T.loc[n, "B"], marker=mk, s=120, color="k", label=n)
        ax.set_xlabel("A: ai_accelerationism"); ax.set_ylabel("B: pro_trisolarans"); ax.set_title(tag); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUT / "superposition_bahn.png", dpi=150); plt.show()
    for tag in tags:
        mid = f"{tag}__w_+0.50"
        if mid in J:
            print(f"\n6 {tag}, Mitte im Gewichtsraum: die am höchsten bewerteten Antworten je Verhalten")
            for k in ("A", "B"):
                rs = sorted([r for r in J[mid]["responses"][k] if r.get("score") is not None], key=lambda r: -r["score"])[:3]
                for r in rs:
                    print(f"   [{k} {r['score']}] {r['response'][:220]!r}")
save_json(OUT / "superposition_summary.json", summary)
print("Fertig. Zusammenfassung in", OUT / "superposition_summary.json")
if USE_DRIVE:
    drive.flush_and_unmount()